In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

import torch.nn as nn
import torch.optim as optim
import math
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
import datetime

import random
from skopt import BayesSearchCV
from sklearn.metrics import r2_score
from sklearn.model_selection import PredefinedSplit
from itertools import product

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
data = pd.read_excel('../Data/New_data11.xlsx')
data_pop = pd.read_excel('../Data/population_2013_2072.xlsx')

In [5]:
for i in range(2013,2025):
    data_pop.loc[data_pop['year']==i,0] = data.loc[data['Year']==i,'Population_0'].iloc[0]
    data_pop.loc[data_pop['year']==i,40] = data.loc[data['Year']==i,'Population_40'].iloc[0]
    data_pop.loc[data_pop['year']==i,60] = data.loc[data['Year']==i,'Population_60'].iloc[0]

In [6]:
Climber = pd.read_excel('../Data/Climber.xlsx')

In [7]:
chuseok_dates = pd.read_excel('../Data/추석날짜.xlsx')

In [8]:
chuseok_week = []
# 각 해의 추석 날짜가 몇 번째 주에 속하는지 계산
for date_str in chuseok_dates['date'].values:
    chuseok_date = date_str.astype('datetime64[s]').astype(datetime.datetime)
    chuseok_week.append(chuseok_date.isocalendar()[1])  # ISO 캘린더에서 주차 계산
print(chuseok_week)

[37, 39, 37, 40, 39, 37, 40, 38, 36, 39, 38, 41, 39, 37, 40, 38, 37, 40, 38, 36, 39, 37, 40, 39, 37, 39, 38, 37, 39, 38, 40, 39, 37, 40, 39, 36, 39, 38, 36, 39, 38, 40, 38, 37, 40, 38, 37, 39, 37, 40, 39, 38, 39, 38, 37, 39, 38]


In [9]:
data['Chuseok'] = 0

In [10]:
for j, i in enumerate(range(2014,2025)):
    data.loc[(data['Year']==i) & (data['Week']>=chuseok_week[j]-2) & (data['Week']<=chuseok_week[j]+2), 'Chuseok']=1

In [11]:
data['Elder_cases'] = data['Cases_60']
data['Elder_incidence'] = 0.0

data['Nonelder_cases'] = data['Cases_0']+data['Cases_40']
data['Nonelder_incidence'] = 0.0

In [12]:
for i in range(2013,2025):
    data.loc[data['Year']==i,'Elder_incidence'] = 1000000*data.loc[data['Year']==i,'Elder_cases']/data_pop.loc[data_pop['year']==i,60].values[0]
    data.loc[data['Year']==i,'Nonelder_incidence'] = 1000000*data.loc[data['Year']==i,'Nonelder_cases']/(data_pop.loc[data_pop['year']==i,40].values[0] + data_pop.loc[data_pop['year']==i,0].values[0])

In [13]:
observed_year_incidence = pd.DataFrame(columns=['year','Total incidence','Elder incidence','Nonelder incidence'])
observed_year_cases = pd.DataFrame(columns=['year','Total cases','Elder cases','Nonelder cases'])
for num, i in enumerate(range(2015,2025)):
    observed_year_incidence.loc[num,'year'] = i
    observed_year_cases.loc[num,'year'] = i
    observed_year_cases.loc[num,'Total cases'] = data.loc[(data['Year']==i),'Cases'].sum()
    observed_year_incidence.loc[num,'Total incidence'] = 1000000*observed_year_cases.loc[num,'Total cases']/data_pop.loc[data_pop['year']==i,[0,40,60]].sum(axis=1).values[0]
    observed_year_incidence.loc[num,'Elder incidence'] = data.loc[(data['Year']==i),'Elder_incidence'].sum()
    observed_year_incidence.loc[num,'Nonelder incidence'] = data.loc[(data['Year']==i),'Nonelder_incidence'].sum()
    observed_year_cases.loc[num,'Elder cases'] = data.loc[(data['Year']==i),'Elder_cases'].sum()
    observed_year_cases.loc[num,'Nonelder cases'] = data.loc[(data['Year']==i),'Nonelder_cases'].sum()

In [14]:
observed_year_incidence['rate']=observed_year_incidence['Elder incidence']/observed_year_incidence['Nonelder incidence']

In [15]:
observed_year_incidence

,year,Total incidence,Elder incidence,Nonelder incidence,rate
0,2015,1.548566,5.811451,0.599191,9.698835
1,2016,3.221536,11.485501,1.278144,8.986078
2,2017,5.295753,18.365782,2.042529,8.991686
3,2018,5.020834,17.0151,1.861381,9.141117
4,2019,4.307945,14.668272,1.409221,10.408781
5,2020,4.68784,14.540912,1.730624,8.402119
6,2021,3.322417,10.368342,1.047658,9.89669
7,2022,3.735057,11.424875,1.115628,10.24076
8,2023,3.828853,11.82998,0.946905,12.493313
9,2024,3.30428,10.106488,0.744646,13.572198


In [16]:
data['Weekly hiker'] = data['Weekly hiker']*(data['Population_60']/data['Population'])

In [17]:
start_year = 2015
end_year = 2023

data_train = data[(data['Year']>=start_year) & (data['Year']<end_year)]
data_test = data[data['Year']>=end_year]

In [18]:
def make_dataset_D(x_data, y_data, window_size):
    x_list = []
    y_list = []
    for i in range(len(x_data) - window_size+1):
        x_list.append(np.array(x_data.iloc[i:i+window_size]))
        y_list.append(np.array(y_data.iloc[i+window_size-1]))
    x_list = np.array(x_list)
    y_list = np.array(y_list).reshape(-1)
    return x_list, y_list

In [19]:
features = ['tem','rain', 'hum', 'Chuseok', 'Weekly hiker', 'Tick Density']

In [20]:
target = ['Elder_incidence']

In [21]:
import random
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [22]:
test_num = 105
valid_num = 52

seed_value = 42

In [23]:
class LSTM(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=2,
                 dropout=0.2):
        super(LSTM, self).__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=len(features),
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc1 = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc2 = nn.Linear(hidden_dim//2, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        h0 = torch.zeros(
            self.num_layers,
            x.shape[0],
            self.hidden_dim,
            device=x.device
        )

        c0 = torch.zeros(
            self.num_layers,
            x.shape[0],
            self.hidden_dim,
            device=x.device
        )

        x, _ = self.lstm(x, (h0, c0))

        x = x[:, -1, :]

        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)

        return x

In [24]:
def train_LSTM(X_train_L, y_train_L, X_valid_L, y_valid_L,
              hidden_dim=64, num_layers=2, dropout=0.2, lr=0.0001,
              epochs=800):
    set_seed(42)
    model = LSTM(hidden_dim, num_layers, dropout).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss_train_record_L=[]
    loss_valid_record_L=[]
    
    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader_L:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
          
        model.eval()
        with torch.no_grad():
            LSTM_pred_train = model(X_train_L.to(device))
            loss_train = criterion(LSTM_pred_train,y_train_L.to(device))
            LSTM_pred_valid = model(X_valid_L.to(device))
            loss_valid = criterion(LSTM_pred_valid,y_valid_L.to(device))
    
            loss_train_record_L.append(loss_train.item())
            loss_valid_record_L.append(loss_valid.item())
    
        if (epoch+1) % 20 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], loss: {loss_train:.5f}, loss_valid: {loss_valid:.5f}')    


    best_idx = min(range(len(loss_valid_record_L)), key=lambda i: loss_valid_record_L[i])
    
    set_seed(42)
    model = LSTM(hidden_dim, num_layers, dropout).to(device)
    criterion = nn.MSELoss()  # 평균 제곱 오차
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(best_idx+1):
        model.train()
        for inputs, targets in train_loader_total_L:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 역전파 및 최적화
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
      
    model.eval()
    with torch.no_grad():
        LSTM_pred_test = model(X_test_L.to(device))
        loss_test = criterion(LSTM_pred_test, y_test_L.to(device))

    return model, loss_test, best_idx

In [26]:
param_grid = {
    "hidden_dim": [64, 128],
    "num_layers": [2, 3],
    "dropout": [0.2, 0.3, 0.4],
    "lr": [0.0002, 0.0001, 0.00005]
}

In [28]:
best_models_info = []

In [29]:
for window_size in range(2,10):
    best_test_loss = float("inf")
    best_model = None
    best_params = None
    data_temp = data_train[(data_train['Year']>start_year) | (data_train['Week']>53-window_size+1)]
    data_temp.index = range(len(data_temp))
    data_test.index = range(len(data_test))
    X = pd.concat([data_temp.loc[:, features],data_test.loc[:, features]])
    Y = pd.concat([data_temp[target],data_test[target]])
    X.index=range(len(X))
    Y.index=range(len(Y))
    X_max = X[:-test_num].max()
    X_min = X[:-test_num].min()
    X_s = (X-X_min)/(X_max-X_min)
    
    train_num = len(X) - valid_num - test_num - window_size + 1
    temp_X = X_s.copy()
    temp_Y = Y.copy()
    
    X_w, Y_w = make_dataset_D(X_s, Y, window_size)
    
    X_train_L = torch.Tensor(X_w[:train_num])
    X_valid_L = torch.Tensor(X_w[train_num:train_num+valid_num])
    X_test_L = torch.Tensor(X_w[-test_num:])
    
    y_train_L = torch.Tensor(Y_w[:train_num]).reshape(-1,1)
    y_valid_L = torch.Tensor(Y_w[train_num:train_num+valid_num]).reshape(-1,1)
    y_test_L = torch.Tensor(Y_w[-test_num:]).reshape(-1,1)
    
    X_train_total_L = torch.cat([X_train_L, X_valid_L], dim=0)
    y_train_total_L = torch.cat([y_train_L, y_valid_L], dim=0)
    
    train_dataset_L = TensorDataset(X_train_L, y_train_L)
    train_loader_L = DataLoader(train_dataset_L, batch_size=16, pin_memory=True)

    train_dataset_total_L = TensorDataset(X_train_total_L, y_train_total_L)
    train_loader_total_L = DataLoader(train_dataset_total_L, batch_size=16, pin_memory=True)
    window_results = []

    for hidden_dim, num_layers, dropout, lr in product(
        param_grid["hidden_dim"],
        param_grid["num_layers"],
        param_grid["dropout"],
        param_grid["lr"]
    ):

        model, test_loss, best_idx = train_LSTM(
            X_train_L, y_train_L,
            X_valid_L, y_valid_L,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            lr=lr,
            epochs=1000
        )
        result = {
            "window_size": window_size,
            "hidden_dim": hidden_dim,
            "num_layers": num_layers,
            "dropout": dropout,
            "lr": lr,
            "test_mse": test_loss,
            "best_idx": best_idx
        }
        window_results.append(result)
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_model = model
            best_params = result

    model_path = f"./hyperparameter_4/LSTM_model_Regression_{window_size}.pth"

    torch.save({
        "window_size": window_size,
        "model_state_dict": best_model.state_dict(),
        "best_params": best_params,
        "test_mse": best_test_loss
    }, model_path)

    best_models_info.append(best_params)
    
best_models_df = pd.DataFrame(best_models_info)
best_models_df = best_models_df.sort_values("test_mse")

best_models_df.to_excel(
    f"./hyperparameter_4/best_lstm_by_window.xlsx",
    index=False
)

Epoch [20/1000], loss: 0.07490, loss_valid: 0.03557
Epoch [40/1000], loss: 0.06935, loss_valid: 0.03460
Epoch [60/1000], loss: 0.06762, loss_valid: 0.03522
Epoch [80/1000], loss: 0.06591, loss_valid: 0.03615
Epoch [100/1000], loss: 0.06414, loss_valid: 0.03776
Epoch [120/1000], loss: 0.06169, loss_valid: 0.04150
Epoch [140/1000], loss: 0.05998, loss_valid: 0.03970
Epoch [160/1000], loss: 0.05666, loss_valid: 0.04027
Epoch [180/1000], loss: 0.05364, loss_valid: 0.04456
Epoch [200/1000], loss: 0.05143, loss_valid: 0.05454
Epoch [220/1000], loss: 0.04744, loss_valid: 0.04618
Epoch [240/1000], loss: 0.04504, loss_valid: 0.05386
Epoch [260/1000], loss: 0.04106, loss_valid: 0.05247
Epoch [280/1000], loss: 0.03849, loss_valid: 0.05802
Epoch [300/1000], loss: 0.04026, loss_valid: 0.04453
Epoch [320/1000], loss: 0.03579, loss_valid: 0.05930
Epoch [340/1000], loss: 0.03690, loss_valid: 0.04458
Epoch [360/1000], loss: 0.03449, loss_valid: 0.05808
Epoch [380/1000], loss: 0.03926, loss_valid: 0.036